# Promosyonlar Rossmann satışlarını ne kadar artırıyor?

**Çalışma sorusu:** Rossmann mağazalarındaki promosyonlar günlük satışları gerçekten ne kadar artırıyor ve bu etki her mağaza tipinde aynı mı?

**Veri kaynağı:** Kaggle, *Rossmann Store Sales* yarışması: https://www.kaggle.com/competitions/rossmann-store-sales (lisans/kullanım koşulları için Kaggle'daki *Rules* sayfasına bakınız).

**Bu defterin akışı:** veri yükleme → temizleme → keşifsel analiz → bulgular → sitede kullanılan özet dosyaların üretimi.

> Bu defteri baştan sona çalıştırmak için: *Kernel → Restart & Run All*.

## 1. Veri yükleme

In [12]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

RAW = Path('.')            # ham veri (Kaggle'dan indirilir, depoya yüklenmez)
OUT = Path('site_data')           # sitenin okuyacağı özet dosyalar
OUT.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / 'train.csv', low_memory=False, parse_dates=['Date'])
store = pd.read_csv(RAW / 'store.csv')

print('train:', train.shape, '| store:', store.shape)
print('Tarih aralığı:', train['Date'].min().date(), '→', train['Date'].max().date())
print('Mağaza sayısı:', train['Store'].nunique())
train.head()

train: (1017209, 9) | store: (1115, 10)
Tarih aralığı: 2013-01-01 → 2015-07-31
Mağaza sayısı: 1115


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


**Sütunlar:** `Sales` günlük ciro (€), `Customers` günlük müşteri sayısı, `Open` mağaza açık mı (1/0), `Promo` o gün promosyon var mı (1/0), `StateHoliday`/`SchoolHoliday` tatil bilgisi. `store.csv` dosyasında her mağazanın tipi (`StoreType`: a, b, c, d), ürün çeşidi (`Assortment`) ve rakip bilgisi bulunur.

## 2. Temizleme

In [13]:
print('Eksik değerler (train):')
print(train.isna().sum()[train.isna().sum() > 0] if train.isna().sum().sum() else 'yok')
print()
print('Eksik değerler (store):')
print(store.isna().sum()[store.isna().sum() > 0])
print()
print('Kapalı gün sayısı:', (train['Open'] == 0).sum())
print('Açık ama satışı 0 olan gün sayısı:', ((train['Open'] == 1) & (train['Sales'] == 0)).sum())
print('Promosyonlu günlerin haftanın hangi günlerine düştüğü:')
print(train.groupby('DayOfWeek')['Promo'].mean().round(2))

Eksik değerler (train):
yok

Eksik değerler (store):
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

Kapalı gün sayısı: 172817
Açık ama satışı 0 olan gün sayısı: 54
Promosyonlu günlerin haftanın hangi günlerine düştüğü:
DayOfWeek
1    0.54
2    0.53
3    0.53
4    0.53
5    0.53
6    0.00
7    0.00
Name: Promo, dtype: float64


### Temizleme kararları ve gerekçeleri

1. **Kapalı günler çıkarıldı** (`Open == 0`). Mağaza kapalıyken satış doğal olarak 0'dır; bunlar promosyonun etkisini ölçmeye katılırsa ortalamayı yapay olarak düşürür.
2. **Açık olduğu hâlde satışı 0 görünen günler çıkarıldı.** Bunlar büyük olasılıkla kayıt hatası ya da fiilen satış yapılmayan günlerdir. Sayıları çok küçük olduğu için sonucu değiştirmez.
3. **Yalnızca pazartesi–cuma günleri karşılaştırıldı.** Tablodaki kontrol, promosyonların *sadece hafta içi* (pazartesi–cuma) yapıldığını gösteriyor. Cumartesi ve pazar günleri hiç promosyon yok. Cumartesiler yüksek satışlı günler olduğu için, "promosyonsuz" grubuna cumartesi ve pazarı dahil etmek karşılaştırmayı adaletsiz hâle getirir ve etkinin mağaza tipine göre sıralamasını bile değiştirebilir. Bu yüzden iki grubu **aynı gün tipinde (hafta içi)** karşılaştırıyoruz.
4. `store.csv`'deki eksik değerler (rakip ve Promo2 bilgileri) bu soruyu cevaplamak için kullanılmadığından **doldurulmadı**.
5. Aykırı değerler **silinmedi**; ortalamalar yanında medyanlar da kontrol edildi (aşağıda).

In [14]:
df = train[(train['Open'] == 1) & (train['Sales'] > 0)].copy()
df = df.merge(store[['Store', 'StoreType', 'Assortment']], on='Store', how='left')

weekday = df[df['DayOfWeek'].between(1, 5)].copy()   # pazartesi–cuma
weekday['Month'] = weekday['Date'].dt.to_period('M').dt.to_timestamp()

print('Temizleme sonrası satır sayısı:', len(df))
print('Hafta içi (karşılaştırmada kullanılan) satır sayısı:', len(weekday))

Temizleme sonrası satır sayısı: 844338
Hafta içi (karşılaştırmada kullanılan) satır sayısı: 696693


## 3. Keşifsel analiz ve bulgular

### Bulgu 1: Promosyonlu günlerde satışlar belirgin biçimde yüksek

Aynı gün tipinde (hafta içi) ortalama günlük satışa bakıyoruz. *Medyan*, değerleri küçükten büyüğe sıraladığımızda ortadaki değerdir ve uç değerlerden etkilenmez; ortalamayı onunla da kontrol ediyoruz.

In [15]:
g = weekday.groupby('Promo')['Sales'].agg(['mean', 'median', 'count'])
g.index = ['Promosyonsuz', 'Promosyonlu']
print(g.round(0))

no_p = weekday.loc[weekday['Promo'] == 0, 'Sales'].mean()
yes_p = weekday.loc[weekday['Promo'] == 1, 'Sales'].mean()
overall_lift = yes_p / no_p - 1
print()
print(f'Ortalama artış: %{overall_lift*100:.1f}')

                mean  median   count
Promosyonsuz  5929.0  5465.0  319818
Promosyonlu   8229.0  7650.0  376875

Ortalama artış: %38.8


**Yorumum (Bulgu 1):** Promosyonlu günlerin cirosu promosyonsuz günlerin cirosuna kıyasla oldukça fazla. Bu da bize promosyonun ciroya olumlu bir katkısı olduğunu gösteriyor.

### Bulgu 2: Artış, müşteri sayısından da, müşteri başı harcamadan da geliyor

Satış = müşteri sayısı × müşteri başı ortalama harcama. Promosyon hangisini artırıyor?

In [16]:
weekday['SalesPerCustomer'] = weekday['Sales'] / weekday['Customers'].replace(0, np.nan)
b = weekday.groupby('Promo').agg(Musteri=('Customers', 'mean'), Sepet=('SalesPerCustomer', 'mean'))
b.index = ['Promosyonsuz', 'Promosyonlu']
print(b.round(2))
print()
print(f"Müşteri sayısı artışı: %{(b.loc['Promosyonlu','Musteri']/b.loc['Promosyonsuz','Musteri']-1)*100:.1f}")
print(f"Müşteri başı harcama artışı: %{(b.loc['Promosyonlu','Sepet']/b.loc['Promosyonsuz','Sepet']-1)*100:.1f}")

              Musteri  Sepet
Promosyonsuz   705.07   8.83
Promosyonlu    844.48  10.18

Müşteri sayısı artışı: %19.8
Müşteri başı harcama artışı: %15.3


### Bulgu 3: Etki mağaza tipine göre farklı mı?

In [17]:
t = weekday.groupby(['StoreType', 'Promo'])['Sales'].mean().unstack()
t.columns = ['Promosyonsuz', 'Promosyonlu']
t['Artis_yuzde'] = (t['Promosyonlu'] / t['Promosyonsuz'] - 1) * 100
t['Magaza_sayisi'] = store.groupby('StoreType')['Store'].nunique()
t.round(1)

,Promosyonsuz,Promosyonlu,Artis_yuzde,Magaza_sayisi
StoreType,,,,
a,6029.2,8304.5,37.7,602
b,9426.8,11311.8,20.0,17
c,5766.7,8042.8,39.5,148
d,5637.9,8018.0,42.2,348


**Yorumum (Bulgu 3):** Promosyon etkisi mağaza tipine göre değişiyor. a, c ve d tiplerinde artış birbirine yakın (yaklaşık %38–42), b tipinde ise belirgin biçimde daha düşük (%20). Ancak b tipinde yalnızca 17 mağaza bulunduğu için bu sonucu dikkatli yorumlamak gerekir.

**Dikkat:** `b` tipi yalnızca 17 mağazadan oluşuyor. Bu yüzden onun için çıkan sonuç, diğer tiplere göre daha az güvenilirdir ve sitede bu belirtilmelidir.

### Bulgu 4: Tek bir mağazada değil, neredeyse her mağazada artış var

Her mağaza için ayrı ayrı promosyonlu ve promosyonsuz hafta içi ortalamalarını karşılaştırıyoruz.

In [18]:
s = weekday.groupby(['Store', 'Promo'])['Sales'].mean().unstack()
s.columns = ['Promosyonsuz', 'Promosyonlu']
s['Artis_yuzde'] = (s['Promosyonlu'] / s['Promosyonsuz'] - 1) * 100
s = s.reset_index().merge(store[['Store', 'StoreType', 'Assortment']], on='Store')

print('Mağaza sayısı:', len(s))
print('Satışı düşen mağaza sayısı:', (s['Artis_yuzde'] < 0).sum())
print(s['Artis_yuzde'].describe().round(1))

Mağaza sayısı: 1115
Satışı düşen mağaza sayısı: 0
count    1115.0
mean       39.9
std        12.7
min         1.9
25%        31.2
50%        39.5
75%        47.6
max       102.1
Name: Artis_yuzde, dtype: float64


**Yorumum (Bulgu 4):** 1.115 mağazanın hiçbirinde promosyonlu günlerde satış düşmedi, yani her mağazada promosyonlu günlerde satışlar daha yüksek çıktı. Ancak artış yüzdeleri mağazalara göre farklı. Bu yüzden tüm mağazaların ortalamasına bakarak promosyonlu günlerde satışların yaklaşık %39 daha yüksek olduğunu söyleyebiliriz.

### Bulgu 5: Etki haftanın günlerine göre değişiyor

In [19]:
d = weekday.groupby(['DayOfWeek', 'Promo'])['Sales'].mean().unstack()
d.columns = ['Promosyonsuz', 'Promosyonlu']
d['Artis_yuzde'] = (d['Promosyonlu'] / d['Promosyonsuz'] - 1) * 100
d.index = ['Pazartesi', 'Salı', 'Çarşamba', 'Perşembe', 'Cuma']
d.round(1)

,Promosyonsuz,Promosyonlu,Artis_yuzde
Pazartesi,6222.7,9772.9,57.1
Salı,5716.4,8277.6,44.8
Çarşamba,5617.7,7685.7,36.8
Perşembe,5750.9,7598.8,32.1
Cuma,6344.4,7744.4,22.1


### Bulgu 6: Zaman içinde (aylık) fark kalıcı mı?

In [20]:
m = weekday.groupby(['Month', 'Promo'])['Sales'].mean().unstack()
m.columns = ['Promosyonsuz', 'Promosyonlu']
m['Artis_yuzde'] = (m['Promosyonlu'] / m['Promosyonsuz'] - 1) * 100
print(m.round(1).head(), '...')
print('Aylık artışın en düşüğü / en yükseği:', round(float(m['Artis_yuzde'].min()), 1), '/', round(float(m['Artis_yuzde'].max()), 1))

            Promosyonsuz  Promosyonlu  Artis_yuzde
Month                                             
2013-01-01        5455.6       7517.8         37.8
2013-02-01        5560.7       7619.9         37.0
2013-03-01        5631.6       8211.4         45.8
2013-04-01        5607.3       7666.3         36.7
2013-05-01        6282.3       8123.1         29.3 ...
Aylık artışın en düşüğü / en yükseği: 19.7 / 52.7


### Önemli not: ilişki, nedensellik değildir

Rossmann promosyonları rastgele atamıyor; hangi günlerde ve hangi mağazalarda promosyon yapılacağına şirket karar veriyor. Dolayısıyla "promosyon günlerinde satışlar daha yüksek" bulgusu **ilişkilidir**; fakat tüm farkın promosyondan *kaynaklandığını* kanıtlamaz. Örneğin promosyon, zaten yoğun olması beklenen dönemlere denk getirilmiş olabilir. Mağaza içi karşılaştırma (Bulgu 4) ve aynı gün tipi karşılaştırması bu riski azaltır ama tamamen ortadan kaldırmaz.

## 4. Sitede kullanılan özet dosyaların üretimi

In [21]:
# 1) Genel sonuçlar
overall = {
    'promosyonsuz_ort': round(float(no_p), 0),
    'promosyonlu_ort': round(float(yes_p), 0),
    'artis_yuzde': round(float(overall_lift * 100), 1),
    'musteri_artis_yuzde': round(float((b.loc['Promosyonlu','Musteri']/b.loc['Promosyonsuz','Musteri']-1)*100), 1),
    'sepet_artis_yuzde': round(float((b.loc['Promosyonlu','Sepet']/b.loc['Promosyonsuz','Sepet']-1)*100), 1),
    'magaza_sayisi': int(len(s)),
    'dusen_magaza_sayisi': int((s['Artis_yuzde'] < 0).sum()),
    'veri_baslangic': str(train['Date'].min().date()),
    'veri_bitis': str(train['Date'].max().date()),
    'analiz_edilen_gun_sayisi': int(len(weekday)),
}
(OUT / 'genel.json').write_text(json.dumps(overall, ensure_ascii=False, indent=2), encoding='utf-8')

# 2) Mağaza tipine göre
tt = t.reset_index().rename(columns={'StoreType': 'magaza_tipi'})
tt.round(1).to_csv(OUT / 'magaza_tipi.csv', index=False)

# 3) Haftanın günlerine göre
dd = d.reset_index().rename(columns={'index': 'gun'})
dd.round(1).to_csv(OUT / 'hafta_gunu.csv', index=False)

# 4) Aylık
mm = m.reset_index().rename(columns={'Month': 'ay'})
mm['ay'] = mm['ay'].dt.strftime('%Y-%m')
mm.round(1).to_csv(OUT / 'aylik.csv', index=False)

# 5) Mağaza bazında (okuyucu keşfi için)
ss = s.rename(columns={'Store': 'magaza', 'StoreType': 'magaza_tipi', 'Assortment': 'cesit'})
ss.round(1).to_csv(OUT / 'magaza_bazinda.csv', index=False)

# 6) Müşteri sayısı ve müşteri başı harcama
bb = b.reset_index().rename(columns={'index': 'grup'})
bb.round(2).to_csv(OUT / 'musteri_sepet.csv', index=False)

import os
toplam = sum(os.path.getsize(OUT / f) for f in os.listdir(OUT))
print('Üretilen dosyalar:', sorted(os.listdir(OUT)))
print(f'Toplam boyut: {toplam/1024:.1f} KB (sınır: 5 MB)')

Üretilen dosyalar: ['aylik.csv', 'genel.json', 'hafta_gunu.csv', 'magaza_bazinda.csv', 'magaza_tipi.csv', 'musteri_sepet.csv']
Toplam boyut: 31.3 KB (sınır: 5 MB)
